# Week 2 · Class 2 — NumPy essentials & pandas fundamentals 🐼

**Plan for today (120 min)**

| Part | Topic | Time |
|---|---|---|
| 1 | NumPy essentials: arrays, shapes, operations, masks | 20 min |
| 2 | pandas fundamentals: Series, DataFrames, selecting, cleaning, grouping, combining | 90 min |
| 3 | Lambda functions & `.apply`: answering questions with custom rules | 10 min |

**Goal:** you won't learn *every* method today. You'll learn the **core vocabulary** of NumPy and pandas, so that you can
read the documentation, search for what you need, and solve Saturday's labs on your own.

After this class you should be able to:
- create NumPy arrays, check their `shape`, reshape/transpose them, compute statistics and label values with conditions
- create pandas `Series` and `DataFrame`s from lists and dictionaries, and read a CSV file
- select rows and columns (`[]`, `.loc`, `.iloc`, boolean filters) and compute statistics
- detect missing values, fix column types, set an index
- summarise data with `groupby`, and combine tables with `concat`, `merge` and `join`
- write `lambda` functions and use `.apply` to create new columns from your own rules

## 🧭 How to find what you don't know

Nobody remembers every method. Professionals use these tricks all the time:

- `np.zeros?` or `help(np.zeros)` → shows the documentation of a function inside the notebook
- Type `df.` and press **Tab** → lists every method available on your object
- Official docs: [NumPy reference](https://numpy.org/doc/stable/reference/index.html) · [pandas API reference](https://pandas.pydata.org/docs/reference/index.html)
- Searching "pandas how to ..." works very well — the answer is usually one method name away

---
# Part 1 — NumPy essentials (20 min) 🔢

**NumPy** is the library for working with numbers in Python. Its central object is the **array**: a grid of values that all have the **same type**.

Why not just Python lists?
- arrays are **much faster** for maths on many numbers
- you can do maths on the whole array at once (no loops needed)
- pandas is built on top of NumPy: every pandas column is a NumPy array underneath

In [1]:
# Import NumPy with its standard nickname "np"
import numpy as np

# Which version do we have?
print(np.__version__)

2.5.3


## 1.1 Creating arrays and inspecting them

The 4 attributes you'll check all the time:

| Attribute | Meaning |
|---|---|
| `.shape` | size along each dimension, e.g. `(2, 3)` = 2 rows, 3 columns |
| `.ndim` | number of dimensions (1 = vector, 2 = matrix, 3+ = tensor) |
| `.size` | total number of elements |
| `.dtype` | type of the elements (`int64`, `float64`, ...) |

In [2]:
# A 1D array (a vector) from a list
v = np.array([2, 3, 5])
print(v, v.shape, v.ndim, v.size, v.dtype)

# A 2D array (a matrix) from a list of lists: each inner list is a row
m = np.array([[1, 2, 3],
              [4, 5, 6]])
print(m)
print("shape:", m.shape, "| ndim:", m.ndim, "| size:", m.size)

[2 3 5] (3,) 1 3 int64
[[1 2 3]
 [4 5 6]]
shape: (2, 3) | ndim: 2 | size: 6


### Ready-made arrays

Very often you don't type values by hand: you ask NumPy to build an array of a given **shape**.
The shape is always passed as a **tuple**, e.g. `(2, 3, 4)`.

In [3]:
# Arrays full of 0s or 1s
zeros = np.zeros((2, 3))
ones = np.ones((2, 3, 4))      # 3D: 2 blocks, each of 3 rows x 4 columns
print(zeros)
print(ones.shape)

# An "empty" array: memory is reserved but values are garbage until you fill them
placeholder = np.empty((2, 3))

# Regularly spaced values
print(np.arange(0, 10, 2))      # start, stop (excluded), step
print(np.linspace(0, 1, 5))     # start, stop (included), number of points

[[0. 0. 0.]
 [0. 0. 0.]]
(2, 3, 4)
[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]


### Random arrays

Random data is great for practising and for simulations. The modern way is to create a **random generator** with a seed
(the seed makes the "random" numbers reproducible: same seed → same numbers).

In [4]:
# Create a random generator (seed = 42 makes results reproducible)
rng = np.random.default_rng(42)

# Uniform floats between 0 and 1, with a given shape
a = rng.random((2, 3, 5))
print(a.shape)

# Random integers between 0 and 10 (10 excluded)
print(rng.integers(0, 10, size=8))

# Normal distribution with mean 5 and standard deviation 2
print(rng.normal(5, 2, size=5).round(2))

# You'll also see the older style in tutorials: it still works
print(np.random.rand(2, 3).shape)

(2, 3, 5)
[9 7 3 9 4 3 9 3]
[6.23 7.26 4.77 3.32 3.35]
(2, 3)


## 1.2 Indexing and slicing

Indexing starts at **0**. For a 2D array you write `array[row, column]`.
`:` means "everything", and `start:stop` takes a range (stop excluded).

In [5]:
b = np.arange(20).reshape(4, 5)
print(b)

# One element: row 1, column 3
print(b[1, 3])

# A whole column
print(b[:, 0])

# Rows 1 and 2, all columns
print(b[1:3, :])

# Last two columns (negative indices count from the end)
print(b[:, -2:])

[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]
8
[ 0  5 10 15]
[[ 5  6  7  8  9]
 [10 11 12 13 14]]
[[ 3  4]
 [ 8  9]
 [13 14]
 [18 19]]


## 1.3 Element-wise operations and broadcasting

Operations like `+`, `-`, `*`, `/` work **element by element**: no loop needed.

⚠️ To combine two arrays, their shapes must be **compatible**. Same shape always works.
A single number (a *scalar*) also works: NumPy "broadcasts" it to every element.

In [6]:
x = np.array([1, 2, 3])
y = np.array([10, 20, 30])

print(x + y)      # element-wise sum
print(x * y)      # element-wise product (NOT a matrix product)
print(x * 100)    # scalar broadcast to every element
print(np.sqrt(y), np.log(y).round(2))

[11 22 33]
[10 40 90]
[100 200 300]
[3.16227766 4.47213595 5.47722558] [2.3 3.  3.4]


In [7]:
# What happens with incompatible shapes?
p = np.ones((2, 3))
q = np.ones((3, 2))

# try/except lets us see the error without stopping the notebook
try:
    p + q
except ValueError as error:
    print("Error:", error)

Error: operands could not be broadcast together with shapes (2,3) (3,2) 


## 1.4 Reshape and transpose

- `.reshape(new_shape)` re-arranges the **same elements** into a new shape. The total size must stay the same. Use `-1` to let NumPy compute one dimension.
- `.T` swaps rows and columns of a matrix. For a 3D array it **reverses** all axes.
- `.transpose(axes)` lets you choose the **new order of the axes** yourself.

In [8]:
c = np.arange(12)
print(c.reshape(3, 4))
print(c.reshape(2, -1).shape)    # NumPy computes the -1 → (2, 6)

m = np.array([[1, 2, 3],
              [4, 5, 6]])
print(m.T)                       # (2, 3) → (3, 2)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
(2, 6)
[[1 4]
 [2 5]
 [3 6]]


In [9]:
# Transposing a 3D array: choose where each old axis goes
t = np.ones((5, 2, 3))           # axes: 0 → size 5, 1 → size 2, 2 → size 3

print(t.T.shape)                 # .T reverses the axes → (3, 2, 5)

# We want shape (2, 3, 5): new axis order = old axes (1, 2, 0)
print(t.transpose(1, 2, 0).shape)

(3, 2, 5)
(2, 3, 5)


## 1.5 Statistics on arrays

`.sum()`, `.mean()`, `.min()`, `.max()`, `.std()` work on the whole array, or along one **axis**:
- `axis=0` → collapse the rows (one result per **column**)
- `axis=1` → collapse the columns (one result per **row**)

In [10]:
scores = np.array([[12, 15, 9],
                   [18, 14, 16],
                   [10, 11, 13]])   # 3 students (rows) x 3 exams (columns)

print("overall mean:", scores.mean().round(2))
print("max:", scores.max(), "| min:", scores.min())
print("mean per exam (axis=0):", scores.mean(axis=0).round(2))
print("mean per student (axis=1):", scores.mean(axis=1).round(2))

overall mean: 13.11
max: 18 | min: 9
mean per exam (axis=0): [13.33 13.33 12.67]
mean per student (axis=1): [12.   16.   11.33]


## 1.6 Masks and conditional labels

A comparison like `scores > 12` gives an array of `True`/`False` called a **mask**. You can use it to:
- **select** values: `scores[mask]`
- **count** them: `mask.sum()` (True counts as 1)
- **label** values: `np.where` (one condition) or `np.select` (several conditions)

These are the tools you need to label values in an array without writing loops.

In [11]:
mask = scores > 12
print(mask)
print("values above 12:", scores[mask])
print("how many:", mask.sum())

# np.where(condition, value_if_true, value_if_false)
print(np.where(scores >= 10, "pass", "fail"))

[[False  True False]
 [ True  True  True]
 [False False  True]]
values above 12: [15 18 14 16 13]
how many: 5
[['pass' 'pass' 'fail']
 ['pass' 'pass' 'pass']
 ['pass' 'pass' 'pass']]


In [12]:
# np.select: list of conditions + list of matching values (first True wins)
conditions = [scores < 12,
              (scores >= 12) & (scores < 16),   # use & for "and", with parentheses!
              scores >= 16]
labels = ["C", "B", "A"]

print(np.select(conditions, labels, default="?"))

# Same idea with numbers, and same shape as the input
print(np.select(conditions, [0, 50, 100]))

[['B' 'B' 'C']
 ['A' 'B' 'A']
 ['C' 'C' 'B']]
[[ 50  50   0]
 [100  50 100]
 [  0   0  50]]


### 🏋️ NumPy practice (3 min)

1. Create a random array `d` of shape `(2, 3, 4)` with values between 1 and 2 (hint: `rng.random` gives values in [0, 1)).
2. Store its min, max and mean in `d_min`, `d_max`, `d_mean`.
3. Create `f`, same shape as `d`, containing `0` where `d` equals its minimum, `100` where it equals its maximum, `25` below the mean and `75` above the mean.

In [13]:
d = 1 + rng.random((2, 3, 4))
d_min, d_max, d_mean = d.min(), d.max(), d.mean()

f = np.select([d == d_min, d == d_max, d < d_mean, d > d_mean],
              [0, 100, 25, 75],
              default=50)
print(d.round(2))
print(f)

[[[1.67 1.44 1.83 1.7 ]
  [1.31 1.83 1.8  1.39]
  [1.29 1.68 1.14 1.2 ]]

 [[1.01 1.79 1.66 1.71]
  [1.78 1.46 1.57 1.14]
  [1.11 1.67 1.47 1.57]]]
[[[ 75  25 100  75]
  [ 25  75  75  25]
  [ 25  75  25  25]]

 [[  0  75  75  75]
  [ 75  25  75  25]
  [ 25  75  25  75]]]


---
# Part 2 — pandas fundamentals (90 min) 🐼

**pandas** is the library for **tables** (like Excel or SQL tables) in Python. Two objects to know:

| Object | What it is | Analogy |
|---|---|---|
| `Series` | one column of values with an **index** (labels) | one Excel column |
| `DataFrame` | several Series side by side, sharing the same index | a whole Excel sheet |

In [14]:
import pandas as pd
print(pd.__version__)

3.0.5


## 2.1 Series (8 min)

A Series is a 1D NumPy array **plus labels** (the index). By default the index is `0, 1, 2, ...`.

In [15]:
temps = pd.Series([5.7, 75.2, 74.4, 84.0, 66.5])
print(temps)

# Third value: position 2 (counting starts at 0)
print(temps.iloc[2])

0     5.7
1    75.2
2    74.4
3    84.0
4    66.5
dtype: float64
74.4


In [16]:
# A Series can have custom labels
prices = pd.Series([2.39, 3.39, 8.75], index=["chips", "drink", "burrito"])
print(prices)

print(prices.loc["burrito"])    # .loc → by LABEL
print(prices.iloc[0])           # .iloc → by POSITION

# Statistics work directly
print(prices.mean(), prices.max(), prices.median())

chips      2.39
drink      3.39
burrito    8.75
dtype: float64
8.75
2.39
4.843333333333334 8.75 3.39


💡 **Rule of thumb:** `.loc` uses labels, `.iloc` uses integer positions. When in doubt, be explicit and use one of them.

## 2.2 Creating DataFrames (7 min)

The two most common ways to build a DataFrame by hand:
- from a **dictionary**: keys become column names, lists become columns
- from a **list of lists**: each inner list becomes a **row**

In [17]:
# From a dictionary of columns
people = pd.DataFrame({
    "name": ["Agnes", "Sidi", "Thibault", "Samia", "Henry", "Georges"],
    "age": [28, 37, 43, 33, 29, 57],
    "job": ["web analyst", "sales director", "web analyst",
            "sales director", "web analyst", "developer"],
})
people

,name,age,job
0,Agnes,28,web analyst
1,Sidi,37,sales director
2,Thibault,43,web analyst
3,Samia,33,sales director
4,Henry,29,web analyst
5,Georges,57,developer


In [18]:
# From a list of lists: each sublist is a row
rows = [[53.1, 95.0, 67.5],
        [61.3, 40.8, 30.8],
        [20.6, 73.2, 44.2]]
scores_df = pd.DataFrame(rows)
print(scores_df)

# Rename ALL columns at once by assigning a list
scores_df.columns = ["Score_1", "Score_2", "Score_3"]

# Or rename only some columns with a dictionary {old: new}
scores_df = scores_df.rename(columns={"Score_3": "Final"})
scores_df

      0     1     2
0  53.1  95.0  67.5
1  61.3  40.8  30.8
2  20.6  73.2  44.2


,Score_1,Score_2,Final
0,53.1,95.0,67.5
1,61.3,40.8,30.8
2,20.6,73.2,44.2


In [19]:
# The key attributes of a DataFrame
print(people.shape)      # (rows, columns)
print(people.columns)    # column names
print(people.index)      # row labels
print(people.dtypes)     # type of each column

(6, 3)
Index(['name', 'age', 'job'], dtype='str')
RangeIndex(start=0, stop=6, step=1)
name      str
age     int64
job       str
dtype: object


## 2.3 Reading a file and first look (10 min)

We'll use a real dataset: orders from a restaurant chain (Chipotle). Each row is one item in an order.

`pd.read_csv` reads CSV files. This file is separated by **tabs**, so we pass `sep="\t"`.
It also works with a URL or a local path like `"data/chipotle.tsv"`.

In [20]:
url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv"
orders = pd.read_csv(url, sep="\t")

In [21]:
# First 5 rows (use .tail() for the last ones)
orders.head()

,order_id,quantity,item_name,choice_description,item_price
0,1,1,Chips and Fresh Tomato Salsa,NaN,$2.39
1,1,1,Izze,[Clementine],$3.39
2,1,1,Nantucket Nectar,[Apple],$3.39
3,1,1,Chips and Tomatillo-Green Chili Salsa,NaN,$2.39
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",$16.98


In [22]:
# How big is it?
print(orders.shape)

# Column names, types and number of non-missing values
orders.info()

(4622, 5)
<class 'pandas.DataFrame'>
RangeIndex: 4622 entries, 0 to 4621
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   order_id            4622 non-null   int64
 1   quantity            4622 non-null   int64
 2   item_name           4622 non-null   str  
 3   choice_description  3376 non-null   str  
 4   item_price          4622 non-null   str  
dtypes: int64(2), str(3)
memory usage: 180.7 KB


In [23]:
# Summary statistics of numeric columns
orders.describe()

,order_id,quantity
count,4622.000000,4622.000000
mean,927.254868,1.075725
std,528.890796,0.410186
min,1.000000,1.000000
25%,477.250000,1.000000
50%,926.000000,1.000000
75%,1393.000000,1.000000
max,1834.000000,15.000000


In [24]:
# For text columns: how many different values, and the most frequent ones
print(orders["item_name"].nunique())
orders["item_name"].value_counts().head(5)

50


item_name
Chicken Bowl           726
Chicken Burrito        553
Chips and Guacamole    479
Steak Burrito          368
Canned Soft Drink      301
Name: count, dtype: int64

## 2.4 Selecting data (15 min)

| You want | Syntax |
|---|---|
| one column (→ Series) | `df["col"]` |
| several columns (→ DataFrame) | `df[["col1", "col2"]]` ← note the **double brackets** |
| rows/columns by label | `df.loc[rows, cols]` |
| rows/columns by position | `df.iloc[rows, cols]` |
| rows matching a condition | `df[condition]` |

In [25]:
# One column → Series
people["age"]

0    28
1    37
2    43
3    33
4    29
5    57
Name: age, dtype: int64

In [26]:
# Several columns → DataFrame (double brackets: a list inside [])
people[["name", "age"]]

,name,age
0,Agnes,28
1,Sidi,37
2,Thibault,43
3,Samia,33
4,Henry,29
5,Georges,57


In [27]:
# .iloc: by position → first 3 rows, first 2 columns
print(people.iloc[0:3, 0:2])

# .loc: by label → rows with index 1 to 3 (INCLUDED with .loc!), columns age and job
print(people.loc[1:3, ["age", "job"]])

       name  age
0     Agnes   28
1      Sidi   37
2  Thibault   43
   age             job
1   37  sales director
2   43     web analyst
3   33  sales director


### Boolean filtering

Exactly like NumPy masks: a condition gives a Series of `True`/`False`, and `df[condition]` keeps the `True` rows.

To combine conditions use `&` (and), `|` (or), `~` (not) — and **always put each condition in parentheses**.

In [28]:
# People older than 35
people[people["age"] > 35]

,name,age,job
1,Sidi,37,sales director
2,Thibault,43,web analyst
5,Georges,57,developer


In [29]:
# Web analysts AND older than 28
mask = (people["job"] == "web analyst") & (people["age"] > 28)
people[mask]

,name,age,job
2,Thibault,43,web analyst
4,Henry,29,web analyst


In [30]:
# Several possible values: .isin()
people[people["job"].isin(["developer", "sales director"])]

,name,age,job
1,Sidi,37,sales director
3,Samia,33,sales director
5,Georges,57,developer


In [31]:
# Counting rows that match: len() of the filtered table, or .sum() of the mask
print(len(people[people["age"] > 35]))
print((people["age"] > 35).sum())

# A statistic on the filtered rows only
print(people[people["job"] == "web analyst"]["age"].mean())

3
3
33.333333333333336


## 2.5 Cleaning basics (12 min)

Real data is messy. Before analysing, always check:
1. **missing values**
2. **column types** (numbers stored as text?)
3. **duplicates / identifiers**

In [32]:
# 1. Missing values per column
orders.isna().sum()

order_id                 0
quantity                 0
item_name                0
choice_description    1246
item_price               0
dtype: int64

In [33]:
# Two common options: drop the rows, or fill the gaps
print(orders.dropna().shape)
orders["choice_description"] = orders["choice_description"].fillna("No choice")
orders["choice_description"].isna().sum()

(3376, 5)


np.int64(0)

In [34]:
# 2. item_price is text like "$2.39 " → we can't do maths with it
print(orders["item_price"].head(3))
print(orders["item_price"].dtype)

# Remove "$", remove spaces, convert to float
orders["item_price"] = (orders["item_price"]
                        .str.replace("$", "", regex=False)
                        .str.strip()
                        .astype(float))
print(orders["item_price"].dtype)
orders["item_price"].describe()

0    $2.39 
1    $3.39 
2    $3.39 
Name: item_price, dtype: str
str
float64


count    4622.000000
mean        7.464336
std         4.245557
min         1.090000
25%         3.390000
50%         8.750000
75%         9.250000
max        44.250000
Name: item_price, dtype: float64

In [35]:
# 3. Setting a column as the index
# drop=False keeps the column in the table as well
people_idx = people.set_index("name", drop=False)
people_idx.loc["Samia"]

name             Samia
age                 33
job     sales director
Name: Samia, dtype: object

In [36]:
# Is a column (or a combination of columns) a unique identifier?
print(people["name"].is_unique)
print(orders["order_id"].is_unique)          # no: one order has several items

# For several columns: look for duplicated combinations
print(orders.duplicated(subset=["order_id", "item_name"]).sum())

True
False
330


## 2.6 New columns and statistics (6 min)

In [37]:
# Arithmetic between columns is element-wise (like NumPy)
orders["unit_price"] = orders["item_price"] / orders["quantity"]

# A custom rule on each value: .apply with a lambda function
people["senior"] = people["age"].apply(lambda age: age >= 40)
people

,name,age,job,senior
0,Agnes,28,web analyst,False
1,Sidi,37,sales director,False
2,Thibault,43,web analyst,True
3,Samia,33,sales director,False
4,Henry,29,web analyst,False
5,Georges,57,developer,True


In [38]:
# Column statistics: sum, mean, median, min, max
print("total quantity:", orders["quantity"].sum())
print("total revenue:", orders["item_price"].sum().round(2))
print("price range:", orders["unit_price"].max() - orders["unit_price"].min())

# Sorting
orders.sort_values("item_price", ascending=False).head(3)

total quantity: 4972
total revenue: 34500.16
price range: 10.8


,order_id,quantity,item_name,choice_description,item_price,unit_price
3598,1443,15,Chips and Fresh Tomato Salsa,No choice,44.25,2.95
3480,1398,3,Carnitas Bowl,"[Roasted Chili Corn Salsa, [Fajita Vegetables,...",35.25,11.75
1254,511,4,Chicken Burrito,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",35.00,8.75


## 2.7 GroupBy: split → apply → combine (12 min)

`df.groupby("col")` splits the table into groups with the same value of `col`, applies a calculation to each group and combines the results.

In [39]:
# Average age per job
people.groupby("job")["age"].mean()

job
developer         57.000000
sales director    35.000000
web analyst       33.333333
Name: age, dtype: float64

In [40]:
# Several statistics at once with .agg
people.groupby("job")["age"].agg(["mean", "min", "max", "count"])

,mean,min,max,count
job,,,,
developer,57.000000,57,57,1
sales director,35.000000,33,37,2
web analyst,33.333333,28,43,3


In [41]:
# Real data: top 5 items by revenue
(orders.groupby("item_name")["item_price"]
       .sum()
       .sort_values(ascending=False)
       .head(5))

item_name
Chicken Bowl           7342.73
Chicken Burrito        5575.82
Steak Burrito          3851.43
Steak Bowl             2260.19
Chips and Guacamole    2201.04
Name: item_price, dtype: float64

In [42]:
# Revenue per order, then the average order value
order_totals = orders.groupby("order_id")["item_price"].sum()
print(order_totals.head())
print("average order value:", order_totals.mean().round(2))

order_id
1    11.56
2    16.98
3    12.67
4    21.00
5    13.70
Name: item_price, dtype: float64
average order value: 18.81


## 2.8 Combining tables (12 min)

| Function | What it does | SQL analogy |
|---|---|---|
| `pd.concat([df1, df2])` | stacks tables **on top of each other** (or side by side with `axis=1`) | `UNION` |
| `pd.merge(left, right, on="key", how=...)` | matches rows using a **key column** | `JOIN` |
| `left.join(right)` | like merge, but matches on the **index** | `JOIN` on index |

In [43]:
sales_1 = pd.DataFrame({"sales_id": ["001", "002", "003", "004"],
                        "people_id": [1, 4, 2, 1],
                        "product_id": ["X789", "X999", "X789", "X990"]})
sales_2 = pd.DataFrame({"sales_id": ["005", "006", "007"],
                        "people_id": [0, 3, 2],
                        "product_id": ["X789", "X999", "X789"]})
products = pd.DataFrame({"product_id": ["X789", "X999", "X990"],
                         "product_desc": ["Apple", "Banana", "Orange"]})
customers = pd.DataFrame({"id": [0, 1, 2, 3, 5],
                          "name": ["Paul", "Perrine", "Moussa", "Michel", "Anne"],
                          "age": [None, 67, 24, 76, 47]})

In [44]:
# Stack rows: same columns, one table under the other
sales = pd.concat([sales_1, sales_2])
print(sales)

# The index is repeated (0..3 then 0..2) → ignore_index=True renumbers it
sales = pd.concat([sales_1, sales_2], ignore_index=True)
sales

  sales_id  people_id product_id
0      001          1       X789
1      002          4       X999
2      003          2       X789
3      004          1       X990
0      005          0       X789
1      006          3       X999
2      007          2       X789


,sales_id,people_id,product_id
0,001,1,X789
1,002,4,X999
2,003,2,X789
3,004,1,X990
4,005,0,X789
5,006,3,X999
6,007,2,X789


In [45]:
# axis=1 puts tables SIDE BY SIDE, aligning on the index
pd.concat([sales_1, products], axis=1)

,sales_id,people_id,product_id,product_id,product_desc
0,001,1,X789,X789,Apple
1,002,4,X999,X999,Banana
2,003,2,X789,X990,Orange
3,004,1,X990,NaN,NaN


In [46]:
# Merge on a common column name
sales_products = pd.merge(sales, products, on="product_id")
sales_products

,sales_id,people_id,product_id,product_desc
0,001,1,X789,Apple
1,002,4,X999,Banana
2,003,2,X789,Apple
3,004,1,X990,Orange
4,005,0,X789,Apple
5,006,3,X999,Banana
6,007,2,X789,Apple


### What if keys don't match?

`people_id` 4 has no customer, and customer 5 bought nothing. The `how` argument decides what to keep:

| `how=` | keeps |
|---|---|
| `"inner"` (default) | only keys present in **both** tables |
| `"left"` | all rows of the left table |
| `"right"` | all rows of the right table |
| `"outer"` | all rows of both, with `NaN` where data is missing |

When the key columns have **different names**, use `left_on=` and `right_on=`.

In [47]:
inner = pd.merge(sales_products, customers, left_on="people_id", right_on="id", how="inner")
outer = pd.merge(sales_products, customers, left_on="people_id", right_on="id", how="outer")
left = pd.merge(sales_products, customers, left_on="people_id", right_on="id", how="left")
print(len(inner), len(outer), len(left))
outer

6 8 7


,sales_id,people_id,product_id,product_desc,id,name,age
0,005,0.0,X789,Apple,0.0,Paul,NaN
1,001,1.0,X789,Apple,1.0,Perrine,67.0
2,004,1.0,X990,Orange,1.0,Perrine,67.0
3,003,2.0,X789,Apple,2.0,Moussa,24.0
4,007,2.0,X789,Apple,2.0,Moussa,24.0
5,006,3.0,X999,Banana,3.0,Michel,76.0
6,002,4.0,X999,Banana,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,5.0,Anne,47.0


In [48]:
# .join matches on the INDEX → set the key as index first
left_tbl = pd.DataFrame({"idx": ["i0", "i1", "i2"], "A": ["a0", "a1", "a2"]})
right_tbl = pd.DataFrame({"idx": ["i1", "i2", "i3"], "C": ["c1", "c2", "c3"]})

left_tbl.set_index("idx").join(right_tbl.set_index("idx"))   # default how="left"

,A,C
idx,,
i0,a0,NaN
i1,a1,c1
i2,a2,c2


## 2.9 A taste of dates and text (5 min)

pandas has two "toolboxes" for special columns:
- `.dt` for dates and times (after converting with `pd.to_datetime`)
- `.str` for text (we already used `.str.replace` and `.str.strip`)

In [49]:
# Text columns: .str methods apply to every value
print(orders["item_name"].str.lower().head(3))
print(orders["item_name"].str.contains("Burrito").sum())   # how many burrito lines?

0    chips and fresh tomato salsa
1                            izze
2                nantucket nectar
Name: item_name, dtype: str
1172


In [50]:
# Dates: text → datetime, then extract parts with .dt
events = pd.DataFrame({"when": ["2026-10-13 18:00", "2026-10-15 18:00", "2026-10-17 10:00"]})
events["when"] = pd.to_datetime(events["when"])

events["weekday"] = events["when"].dt.day_name()
events["hour"] = events["when"].dt.hour
events["next_week"] = events["when"] + pd.to_timedelta(7, unit="D")
events

,when,weekday,hour,next_week
0,2026-10-13 18:00:00,Tuesday,18,2026-10-20 18:00:00
1,2026-10-15 18:00:00,Thursday,18,2026-10-22 18:00:00
2,2026-10-17 10:00:00,Saturday,10,2026-10-24 10:00:00


---
# Part 3 — Lambda functions & `.apply` (10 min) λ

Sometimes pandas has no ready-made method for the rule you need. Then you write the rule as a **function** and ask pandas to
**apply** it to every value of a column.

### What is a lambda?

A `lambda` is a **tiny function without a name**, written in one line. These two are equivalent:

```python
def add_vat(price):
    return price * 1.19

add_vat = lambda price: price * 1.19
```

Read it as: *"take `price`, give back `price * 1.19`"*. Use `lambda` for short one-line rules, and `def` when the rule needs several lines.

In [51]:
# A lambda is a normal function: you can store it and call it
add_vat = lambda price: price * 1.19
print(round(add_vat(10), 2))   # 11.9

# Lambdas can take several inputs
total = lambda price, qty: price * qty
print(total(2.5, 4))

11.9
10.0


### `.apply` on a column

`df["col"].apply(function)` calls the function **once per value** and returns a new column with the results.

In [52]:
# Question 1: how are orders split into price bands?
orders["price_band"] = orders["unit_price"].apply(
    lambda p: "cheap" if p < 5 else ("mid" if p < 10 else "premium"))

orders["price_band"].value_counts()

price_band
mid        1890
cheap      1698
premium    1034
Name: count, dtype: int64

In [53]:
# Question 2: which protein is the most popular in burritos, bowls and tacos?
proteins = ["Chicken", "Steak", "Barbacoa", "Carnitas", "Veggie"]

orders["protein"] = orders["item_name"].apply(
    lambda name: name.split()[0] if name.split()[0] in proteins else "none")

orders["protein"].value_counts()

protein
none        1764
Chicken     1560
Steak        702
Veggie       212
Barbacoa     203
Carnitas     181
Name: count, dtype: int64

### When the rule is longer: use `def`

`.apply` accepts **any** function, not only lambdas. If your rule needs several `if`s, a named function is easier to read.

In [54]:
drinks = ["Canned Soft Drink", "Canned Soda", "Bottled Water", "6 Pack Soft Drink", "Izze", "Nantucket Nectar"]

def item_category(name):
    if name in drinks:
        return "drink"
    elif "Chips" in name:
        return "side"
    else:
        return "main"

orders["category"] = orders["item_name"].apply(item_category)

# Question 3: what share of the revenue comes from each category?
(orders.groupby("category")["item_price"].sum() / orders["item_price"].sum()).round(3)

category
drink    0.041
main     0.850
side     0.109
Name: item_price, dtype: float64

In [55]:
# Question 4: do people pay more when they add guacamole?
orders["has_guac"] = orders["choice_description"].apply(lambda text: "Guacamole" in text)

mains = orders[orders["category"] == "main"]
mains.groupby("has_guac")["unit_price"].mean().round(2)

has_guac
False     8.84
True     11.37
Name: unit_price, dtype: float64

### `.apply` with `groupby`: one rule per group

After a `groupby`, `.apply(lambda group: ...)` receives **all the values of one group** at once.

In [56]:
# Question 5: what share of orders includes at least one drink?
order_has_drink = orders.groupby("order_id")["category"].apply(lambda cats: (cats == "drink").any())
print("orders with a drink:", round(order_has_drink.mean() * 100, 1), "%")

orders with a drink: 32.1 %


💡 **Good habit:** `.apply` runs Python code row by row, so it is slower on big data. If a vectorised version exists
(arithmetic between columns, `.str.contains`, `np.where`...), prefer it. Use `.apply` when you need **custom logic**.

### 🏋️ Your turn (in class)

**A.** Create a column `n_toppings` with the number of choices in `choice_description` (hint: count the commas and add 1; `"No choice"` should give 0). What is the average `unit_price` for each number of toppings?

In [57]:
orders["n_toppings"] = orders["choice_description"].apply(
    lambda text: 0 if text == "No choice" else text.count(",") + 1)

orders.groupby("n_toppings")["unit_price"].mean().round(2)

n_toppings
0      3.00
1      2.53
2      9.01
3      9.07
4      9.26
5      9.58
6      9.95
7     10.43
8     10.87
9     10.13
10     9.74
Name: unit_price, dtype: float64

**B.** Using `order_totals` (revenue per order, from section 2.7), label each order `"small"` (< 15), `"medium"` (15–30) or `"large"` (> 30) with `.apply`. How many orders are in each group?

In [58]:
order_size = order_totals.apply(lambda t: "small" if t < 15 else ("medium" if t <= 30 else "large"))
order_size.value_counts()

item_price
small     852
medium    839
large     143
Name: count, dtype: int64

---
## 🏋️ Practice (at home or if time allows)

Use the `orders` table (already cleaned above).

**1.** How many rows have a `quantity` greater than 1 **and** an `item_price` above 10?

In [59]:
((orders["quantity"] > 1) & (orders["item_price"] > 10)).sum()

np.int64(133)

**2.** What is the average `item_price` of items whose name contains `"Bowl"`?

In [60]:
orders[orders["item_name"].str.contains("Bowl")]["item_price"].mean().round(2)

np.float64(10.4)

**3.** For each `item_name`, compute the total `quantity` sold. Which 3 items sold the most units?

In [61]:
orders.groupby("item_name")["quantity"].sum().sort_values(ascending=False).head(3)

item_name
Chicken Bowl           761
Chicken Burrito        591
Chips and Guacamole    506
Name: quantity, dtype: int64

**4.** Using `sales`, `products` and `customers`: build a table with one row per sale, the product description and the customer name. Keep all sales, even those without a known customer. How many sales have no customer name?

In [62]:
full = (sales.merge(products, on="product_id")
             .merge(customers, left_on="people_id", right_on="id", how="left"))
print(full[["sales_id", "product_desc", "name"]])
print("sales without customer:", full["name"].isna().sum())

  sales_id product_desc     name
0      001        Apple  Perrine
1      002       Banana      NaN
2      003        Apple   Moussa
3      004       Orange  Perrine
4      005        Apple     Paul
5      006       Banana   Michel
6      007        Apple   Moussa
sales without customer: 1


---
## 📋 Cheat sheet

**NumPy**
- create: `np.array`, `np.zeros`, `np.ones`, `np.empty`, `np.arange`, `np.linspace`, `rng = np.random.default_rng(seed)` → `rng.random`, `rng.integers`, `rng.normal`
- inspect: `.shape`, `.ndim`, `.size`, `.dtype`
- reshape: `.reshape`, `.T`, `.transpose(axes)`
- stats: `.sum()`, `.mean()`, `.min()`, `.max()`, `.std()` (with `axis=`)
- conditions: masks `a[a > x]`, `np.where`, `np.select`

**pandas**
- create/read: `pd.Series`, `pd.DataFrame`, `pd.read_csv`
- inspect: `.head()`, `.tail()`, `.shape`, `.info()`, `.describe()`, `.dtypes`, `.value_counts()`, `.nunique()`
- select: `df["col"]`, `df[["a", "b"]]`, `.loc`, `.iloc`, `df[cond]` with `&`, `|`, `~`, `.isin`
- clean: `.isna().sum()`, `.dropna()`, `.fillna()`, `.astype()`, `.rename()`, `.set_index()`, `.is_unique`, `.duplicated()`
- transform: column arithmetic, `.apply(lambda ...)` or `.apply(my_function)`, `groupby(...).apply(...)`, `.sort_values()`
- summarise: `.groupby().agg()`
- combine: `pd.concat`, `pd.merge(how=, on= / left_on= right_on=)`, `.join`
- special columns: `.str.*`, `pd.to_datetime`, `.dt.*`

**Next class:** we go deeper into exploratory data analysis and statistics (mean vs median, spread, outliers).